In [1]:
# Install required packages
!pip install xgboost imbalanced-learn -q

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Setup complete!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

✓ Setup complete!
Pandas: 2.2.2
NumPy: 1.26.4


In [2]:
file_1 = 'Downloads/salmonella_dna/genes_kmers_trim_labels_398.csv'
file_2 = 'Downloads/salmonella_dna/snps_only_labels_trim_398.csv'
file_3 = 'Downloads/salmonella_dna/snps_kmers_trim_labels_398.csv'
file_4 = 'Downloads/salmonella_dna/genes_snps_trim_labels_398.csv'

In [3]:
# Configuration
CONFIG = {
    'n_folds': 6,
    'random_state': 42,
    'smote_strategy': 0.5,  # Oversample resistant to 50% of majority class
    'top_n_features': 30,
    
    # Files to process
    'datasets': {
        'Genes + SNPs': file_1,
        'SNPs Only': file_2,
        'SNPs + K-mers': file_3,
        'Genes + K-mers': file_4
    }
}

print("Configuration:")
for key, value in CONFIG.items():
    if key != 'datasets':
        print(f"  {key}: {value}")

print(f"\nDatasets to process: {len(CONFIG['datasets'])}")

Configuration:
  n_folds: 6
  random_state: 42
  smote_strategy: 0.5
  top_n_features: 30

Datasets to process: 4


In [4]:
def load_and_validate_data(filepath):
    """Load CSV and validate structure"""
    df = pd.read_csv(filepath)
    
    # Separate features and labels
    X = df.drop(columns=['Genome_ID', 'Pefloxacin_Phenotype'])
    y = df['Pefloxacin_Phenotype']
    genome_ids = df['Genome_ID']
    
    # Validate
    assert len(X) == len(y), "Mismatch in samples"
    assert y.isin([0, 1]).all(), "Labels must be 0 or 1"
    
    print(f"  ✓ Loaded: {len(X)} samples × {X.shape[1]} features")
    print(f"    Susceptible (0): {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)")
    print(f"    Resistant (1): {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)")
    
    return X, y, genome_ids, df.columns.tolist()

def get_models(scale_pos_weight):
    """Define all models with class imbalance handling"""
    return {
        'XGBoost': XGBClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            subsample=0.8,
            colsample_bytree=0.5,
            reg_alpha=1,
            reg_lambda=1,
            random_state=CONFIG['random_state'],
            eval_metric='logloss'
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=CONFIG['random_state']
        ),
        'ExtraTrees': ExtraTreesClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=CONFIG['random_state']
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=CONFIG['random_state']
        )
    }

def train_and_evaluate(X, y, models, use_smote=False, dataset_name=""):
    """Train models with cross-validation"""
    
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, 
                          random_state=CONFIG['random_state'])
    
    results = {name: [] for name in models.keys()}
    
    print(f"\n{'='*70}")
    print(f"Training: {dataset_name} {'WITH SMOTE' if use_smote else 'BASELINE'}")
    print(f"{'='*70}")
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Apply SMOTE if requested
        if use_smote:
            smote = SMOTE(sampling_strategy=CONFIG['smote_strategy'], 
                         random_state=CONFIG['random_state'])
            X_train, y_train = smote.fit_resample(X_train, y_train)
        
        print(f"\nFold {fold}/{CONFIG['n_folds']}:")
        print(f"  Train: {len(y_train)} ({(y_train==1).sum()} resistant)")
        print(f"  Test: {len(y_test)} ({(y_test==1).sum()} resistant)")
        
        for model_name, model in models.items():
            # Train
            model.fit(X_train, y_train)
            
            # Predict
            y_pred = model.predict(X_test)
            
            # Metrics
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            
            results[model_name].append({
                'fold': fold,
                'accuracy': acc,
                'precision': prec,
                'recall': rec,
                'f1': f1
            })
            
            print(f"    {model_name:20s} Acc={acc:.3f}, F1={f1:.3f}, Rec={rec:.3f}")
    
    return results

def summarize_results(results):
    """Calculate average metrics across folds"""
    summary = {}
    
    for model_name, scores in results.items():
        df = pd.DataFrame(scores)
        summary[model_name] = {
            'accuracy': df['accuracy'].mean(),
            'accuracy_std': df['accuracy'].std(),
            'precision': df['precision'].mean(),
            'precision_std': df['precision'].std(),
            'recall': df['recall'].mean(),
            'recall_std': df['recall'].std(),
            'f1': df['f1'].mean(),
            'f1_std': df['f1'].std()
        }
    
    return summary

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [7]:
# Store all results
all_results = {}

for dataset_name, filepath in CONFIG['datasets'].items():
    
    print(f"\n\n{'#'*70}")
    print(f"# PROCESSING: {dataset_name}")
    print(f"# File: {filepath}")
    print(f"{'#'*70}")
    
    # Load data
    X, y, genome_ids, feature_names = load_and_validate_data(filepath)
    
    # Calculate scale_pos_weight
    #scale_pos_weight = (y==0).sum() / (y==1).sum()
    
    # Get models
    models = get_models(scale_pos_weight)
    
    # Train baseline (no SMOTE)
    baseline_results = train_and_evaluate(X, y, models, use_smote=False, 
                                         dataset_name=dataset_name)
    
    # Train with SMOTE
    #smote_results = train_and_evaluate(X, y, models, use_smote=True,dataset_name=dataset_name)
    
    # Store results
    all_results[dataset_name] = {
        'baseline': baseline_results,
        'n_features': X.shape[1],
        'n_samples': len(X)
    }

print("\n\n✓ All training complete!")



######################################################################
# PROCESSING: Genes + SNPs
# File: Downloads/salmonella_dna/genes_kmers_trim_labels_398.csv
######################################################################
  ✓ Loaded: 398 samples × 5160 features
    Susceptible (0): 211 (53.0%)
    Resistant (1): 187 (47.0%)

Training: Genes + SNPs BASELINE

Fold 1/6:
  Train: 331 (155 resistant)
  Test: 67 (32 resistant)
    XGBoost              Acc=0.940, F1=0.938, Rec=0.938
    RandomForest         Acc=0.955, F1=0.952, Rec=0.938
    ExtraTrees           Acc=0.940, F1=0.938, Rec=0.938
    GradientBoosting     Acc=0.940, F1=0.938, Rec=0.938

Fold 2/6:
  Train: 331 (156 resistant)
  Test: 67 (31 resistant)
    XGBoost              Acc=0.851, F1=0.844, Rec=0.871
    RandomForest         Acc=0.866, F1=0.862, Rec=0.903
    ExtraTrees           Acc=0.866, F1=0.862, Rec=0.903
    GradientBoosting     Acc=0.851, F1=0.844, Rec=0.871

Fold 3/6:
  Train: 332 (156 resistant)
  Test:

FileNotFoundError: [Errno 2] No such file or directory: 'Downloads/salmonella_dna/snps_only_trim_labels_398.csv'

In [ ]:
# ============================================
# ENHANCED RESULTS COMPARISON WITH ACCURACY
# ============================================

# Create comprehensive comparison table
comparison_data = []
for dataset_name, results in all_results.items():
    baseline_summary = summarize_results(results['baseline'])
    smote_summary = summarize_results(results['smote'])
    
    for model_name in baseline_summary.keys():
        # Baseline metrics
        baseline_f1 = baseline_summary[model_name]['f1']
        baseline_acc = baseline_summary[model_name]['accuracy']
        baseline_recall = baseline_summary[model_name]['recall']
        baseline_precision = baseline_summary[model_name]['precision']
        
        # SMOTE metrics
        smote_f1 = smote_summary[model_name]['f1']
        smote_acc = smote_summary[model_name]['accuracy']
        smote_recall = smote_summary[model_name]['recall']
        smote_precision = smote_summary[model_name]['precision']
        
        # Improvements
        f1_improvement = ((smote_f1 - baseline_f1) / baseline_f1) * 100 if baseline_f1 > 0 else 0
        acc_improvement = ((smote_acc - baseline_acc) / baseline_acc) * 100 if baseline_acc > 0 else 0
        
        comparison_data.append({
            'Dataset': dataset_name,
            'Model': model_name,
            'Features': results['n_features'],
            'Baseline Accuracy': baseline_acc,
            'SMOTE Accuracy': smote_acc,
            'Baseline F1': baseline_f1,
            'SMOTE F1': smote_f1,
            'Baseline Precision': baseline_precision,
            'SMOTE Precision': smote_precision,
            'Baseline Recall': baseline_recall,
            'SMOTE Recall': smote_recall,
            'F1 Improvement (%)': f1_improvement,
            'Accuracy Improvement (%)': acc_improvement
        })

comparison_df = pd.DataFrame(comparison_data)

# ============================================
# DISPLAY NICE TABLE (LIKE YOUR ORIGINAL)
# ============================================

print("\n" + "="*140)
print("COMPLETE RESULTS COMPARISON (WITH ACCURACY)")
print("="*140)

# Format for nice display
display_df = comparison_df[[
    'Dataset', 'Model', 'Features',
    'Baseline Accuracy', 'SMOTE Accuracy',
    'Baseline F1', 'SMOTE F1',
    'F1 Improvement (%)',
    'Baseline Recall', 'SMOTE Recall'
]].copy()

print(display_df.to_string(index=False))

# ============================================
# SAVE TO CSV
# ============================================

csv_filename = 'amr_results_comparison_398.csv'
comparison_df.to_csv(csv_filename, index=False, float_format='%.6f')
print(f"\n✓ Full results saved to: {csv_filename}")

# ============================================
# FIND BEST PERFORMERS (ACROSS ALL ROWS)
# ============================================

print("\n" + "="*140)
print("🏆 BEST PERFORMERS (Comparing ALL Baseline & SMOTE Results)")
print("="*140)

# Create combined metrics for comparison (both baseline and SMOTE)
comparison_df['Best F1'] = comparison_df[['Baseline F1', 'SMOTE F1']].max(axis=1)
comparison_df['Best Accuracy'] = comparison_df[['Baseline Accuracy', 'SMOTE Accuracy']].max(axis=1)
comparison_df['Best Recall'] = comparison_df[['Baseline Recall', 'SMOTE Recall']].max(axis=1)

# Add column to indicate which version achieved best F1
comparison_df['Best F1 Source'] = comparison_df.apply(
    lambda row: 'SMOTE' if row['SMOTE F1'] >= row['Baseline F1'] else 'Baseline', 
    axis=1
)

# Add column to indicate which version achieved best Accuracy
comparison_df['Best Accuracy Source'] = comparison_df.apply(
    lambda row: 'SMOTE' if row['SMOTE Accuracy'] >= row['Baseline Accuracy'] else 'Baseline', 
    axis=1
)

# Find absolute best F1 (across baseline AND SMOTE)
best_f1_row = comparison_df.loc[comparison_df['Best F1'].idxmax()]

print("\n🥇 BEST F1-SCORE (Baseline vs SMOTE):")
print(f"  Dataset: {best_f1_row['Dataset']}")
print(f"  Model: {best_f1_row['Model']}")
print(f"  Best F1: {best_f1_row['Best F1']:.3f} ({best_f1_row['Best F1 Source']})")
print(f"    - Baseline F1: {best_f1_row['Baseline F1']:.3f}")
print(f"    - SMOTE F1: {best_f1_row['SMOTE F1']:.3f}")
print(f"  Accuracy: {best_f1_row['Best Accuracy']:.3f} ({best_f1_row['Best Accuracy Source']})")
print(f"    - Baseline Accuracy: {best_f1_row['Baseline Accuracy']:.3f}")
print(f"    - SMOTE Accuracy: {best_f1_row['SMOTE Accuracy']:.3f}")
print(f"  Recall: {best_f1_row['Best Recall']:.3f}")
print(f"  Features: {best_f1_row['Features']}")

# Find absolute best Accuracy (across baseline AND SMOTE)
best_acc_row = comparison_df.loc[comparison_df['Best Accuracy'].idxmax()]

print("\n🥇 BEST ACCURACY (Baseline vs SMOTE):")
print(f"  Dataset: {best_acc_row['Dataset']}")
print(f"  Model: {best_acc_row['Model']}")
print(f"  Best Accuracy: {best_acc_row['Best Accuracy']:.3f} ({best_acc_row['Best Accuracy Source']})")
print(f"    - Baseline Accuracy: {best_acc_row['Baseline Accuracy']:.3f}")
print(f"    - SMOTE Accuracy: {best_acc_row['SMOTE Accuracy']:.3f}")
print(f"  F1-Score: {best_acc_row['Best F1']:.3f} ({best_acc_row['Best F1 Source']})")
print(f"    - Baseline F1: {best_acc_row['Baseline F1']:.3f}")
print(f"    - SMOTE F1: {best_acc_row['SMOTE F1']:.3f}")
print(f"  Features: {best_acc_row['Features']}")

# Find absolute best Recall (important for clinical use)
best_recall_row = comparison_df.loc[comparison_df['Best Recall'].idxmax()]

print("\n🥇 BEST RECALL (Most Resistant Cases Caught):")
print(f"  Dataset: {best_recall_row['Dataset']}")
print(f"  Model: {best_recall_row['Model']}")
print(f"  Best Recall: {best_recall_row['Best Recall']:.3f}")
print(f"    - Baseline Recall: {best_recall_row['Baseline Recall']:.3f}")
print(f"    - SMOTE Recall: {best_recall_row['SMOTE Recall']:.3f}")
print(f"  F1-Score: {best_recall_row['Best F1']:.3f}")
print(f"  Features: {best_recall_row['Features']}")

# ============================================
# TOP 5 MODELS BY F1-SCORE (ALL VERSIONS)
# ============================================

print("\n" + "="*140)
print("📊 TOP 5 MODELS BY F1-SCORE (Across ALL Baseline & SMOTE Results)")
print("="*140)

top5 = comparison_df.nlargest(5, 'Best F1')[
    ['Dataset', 'Model', 'Best F1', 'Best F1 Source', 'Best Accuracy', 'Features']
]

print(top5.to_string(index=False))

# ============================================
# OVERALL STATISTICS
# ============================================

print("\n" + "="*140)
print("📈 OVERALL STATISTICS (Across ALL Results)")
print("="*140)
print(f"Highest F1-Score: {comparison_df['Best F1'].max():.3f}")
print(f"Highest Accuracy: {comparison_df['Best Accuracy'].max():.3f}")
print(f"Highest Recall: {comparison_df['Best Recall'].max():.3f}")
print(f"Highest Precision: {comparison_df[['Baseline Precision', 'SMOTE Precision']].max().max():.3f}")

print(f"\nAverage F1-Score (All Models): {comparison_df['Best F1'].mean():.3f}")
print(f"Average Accuracy (All Models): {comparison_df['Best Accuracy'].mean():.3f}")

# Count how many models improved with SMOTE vs baseline
improved_with_smote = (comparison_df['SMOTE F1'] > comparison_df['Baseline F1']).sum()
total_models = len(comparison_df)

print(f"\nModels improved with SMOTE: {improved_with_smote}/{total_models} ({improved_with_smote/total_models*100:.1f}%)")

# ============================================
# TARGET ASSESSMENT
# ============================================

print("\n" + "="*140)
print("🎯 TARGET ASSESSMENT (81% F1-Score)")
print("="*140)

if best_f1_row['Best F1'] >= 0.81:
    print(f"✅ TARGET ACHIEVED! F1 = {best_f1_row['Best F1']:.1%}")
    print(f"   Exceeded target by: +{(best_f1_row['Best F1'] - 0.81)*100:.1f} percentage points")
    print(f"   Achieved with: {best_f1_row['Dataset']} + {best_f1_row['Model']} ({best_f1_row['Best F1 Source']})")
else:
    gap = (0.81 - best_f1_row['Best F1']) * 100
    print(f"⚠️ Gap to target: {gap:.1f} percentage points")
    print(f"   Current best: {best_f1_row['Best F1']:.1%}")
    print(f"   From: {best_f1_row['Dataset']} + {best_f1_row['Model']} ({best_f1_row['Best F1 Source']})")



In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Performance Comparison Across All Combinations', fontsize=16, fontweight='bold')

datasets = list(CONFIG['datasets'].keys())

for idx, dataset in enumerate(datasets):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    # Extract data for this dataset
    dataset_data = comparison_df[comparison_df['Dataset'] == dataset]
    
    x = np.arange(len(dataset_data['Model'].unique()))
    width = 0.35
    
    baseline_f1 = dataset_data.groupby('Model')['Baseline F1'].first().values
    smote_f1 = dataset_data.groupby('Model')['SMOTE F1'].first().values
    
    bars1 = ax.bar(x - width/2, baseline_f1, width, label='Baseline', alpha=0.8)
    bars2 = ax.bar(x + width/2, smote_f1, width, label='With SMOTE', alpha=0.8)
    
    # Add target line
    ax.axhline(y=0.81, color='r', linestyle='--', label='Target (81%)', linewidth=2)
    
    ax.set_xlabel('Model', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title(f'{dataset}\n({dataset_data["Features"].iloc[0]} features)', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_data['Model'].unique(), rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Save figure
output_path = 'performance_model_comparison.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
#plt.savefig('performance_comparison.jpeg', dpi=300, bbox_inches='tight')
print("✓ Saved: performance_model_comparison.png")

In [ ]:
# Train final model on best combination
best_dataset = best_row['Dataset']
best_model_name = best_row['Model']

print(f"\n{'='*70}")
print(f"FEATURE IMPORTANCE ANALYSIS")
print(f"Dataset: {best_dataset}")
print(f"Model: {best_model_name}")
print(f"{'='*70}")

# Load best dataset
filepath = CONFIG['datasets'][best_dataset]
X, y, genome_ids, feature_names = load_and_validate_data(filepath)
feature_list = [f for f in feature_names if f not in ['Genome_ID', 'Pefloxacin_Phenotype']]

# Apply SMOTE
smote = SMOTE(sampling_strategy=CONFIG['smote_strategy'], random_state=CONFIG['random_state'])
X_resampled, y_resampled = smote.fit_resample(X, y)

# Train final model
scale_pos_weight = (y==0).sum() / (y==1).sum()
models = get_models(scale_pos_weight)
final_model = models[best_model_name]
final_model.fit(X_resampled, y_resampled)

# Get feature importance
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    print(f"\nTop 20 Most Important Features:")
    print(f"{'Rank':<6} {'Feature':<50} {'Importance':<12}")
    print("-" * 70)
    
    for i in range(min(20, len(indices))):
        idx = indices[i]
        print(f"{i+1:<6} {feature_list[idx]:<50} {importances[idx]:.6f}")
    
    # Visualize top 20
    top_n = 20
    top_indices = indices[:top_n]
    top_features = [feature_list[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    plt.figure(figsize=(10, 8))
    plt.barh(range(top_n), top_importances[::-1])
    plt.yticks(range(top_n), top_features[::-1])
    plt.xlabel('Importance', fontweight='bold')
    plt.title(f'Top {top_n} Features - {best_dataset} ({best_model_name})', 
              fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: feature_importance.png")

In [ ]:
# Export detailed results to CSV
comparison_df.to_csv('training_results_summary.csv', index=False)
print("✓ Saved: training_results_summary.csv")

# Export best model's top features
if hasattr(final_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': [feature_list[i] for i in indices],
        'Importance': importances[indices]
    })
    importance_df.to_csv('feature_importance_ranking.csv', index=False)
    print("✓ Saved: feature_importance_ranking.csv")

